# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/en/latest/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata summary
meta = dataset.metadata
print(f"Dataset name: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Identifier: {meta.identifier}\n")
print(f"Version: {meta.version}\n")
print(f"Date published: {meta.datePublished}\n")
print(f"License: {meta.license}\n")


## 2. Data Overview
Review available record sets, fields, and their IDs. All `@id` fields uniquely identify schema entities in the Croissant dataset.


In [ ]:
# List all record sets in the dataset by their @id and names
print("Record sets available in this dataset:")
for record_set in dataset.metadata.recordSets:
    print(f"- @id: {record_set['@id']}, name: {record_set.get('name', '<no name>')}")

# For demonstration, list fields for each record set using their @id
print("\nFields in each record set:")
for record_set in dataset.metadata.recordSets:
    fields = record_set.get('fields', [])
    print(f"- Record set @id: {record_set['@id']}")
    for field in fields:
        print(f"    - Field @id: {field['@id']}, name: {field.get('name', '<no name>')}, dataType: {field.get('dataType', '<no type>')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
All access should use the `@id` fields gathered above.

In [ ]:
# Collect all record set @id values
record_set_ids = [r['@id'] for r in dataset.metadata.recordSets]
print(f"Record set @id values: {record_set_ids}")

# Load each record set into a DataFrame, using the @id
dataframes = {}
for record_set_id in record_set_ids:
    # Use records(record_set=) with @id
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show column names for the primary record set (first one)
main_record_set_id = record_set_ids[0]
print(f"\nColumns in '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common processing: filter, normalize numerical fields, group by categorical variables. All columns and group keys are referenced by their schema `@id`.


In [ ]:
# Pick a numeric field: find the first numeric (e.g., Integer or Float) field in the primary record set
main_record_set = dataset.metadata.recordSets[0]

# Find a numeric field "@id" and a groupable field "@id"
numeric_field_id = None
group_field_id = None

for field in main_record_set.get('fields', []):
    dtype = str(field.get('dataType', ''))
    if numeric_field_id is None and ('Integer' in dtype or 'Float' in dtype):
        numeric_field_id = field['@id']
    if group_field_id is None and ('location' in field.get('name', '').lower() or 'sex' in field.get('name', '').lower() or 'group' in dtype):
        group_field_id = field['@id']

print(f"Numeric field selected: {numeric_field_id}")
print(f"Group field selected: {group_field_id}")

df = dataframes[main_record_set_id].copy()

# Ensure field is numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean()  # Use the mean as threshold for demonstration
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold:.1f} ({filtered_df.shape[0]} rows):")
print(filtered_df[[numeric_field_id]].head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field_id if it exists
if group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and compare group means if a group field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=12, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f'Distribution of {numeric_field_id}')
plt.show()

if group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(9, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=25)
    plt.show()

## 6. Conclusion
This notebook demonstrates how to load, inspect, and explore a FAIR^2 dataset with Croissant schema using Python and `mlcroissant`. You can continue analysis or build ML workflows referencing variables, record sets, and columns by their unique `@id` fields as shown above.